<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/PipelineStepsPart3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Calving-pen pipeline, part 3 of 3 (long videos)

Three people run parts 1, 2 and 3 at the same time, each on their own Colab (GPU: T4). This part runs **CalvingPen2Oct29, CalvingPen2Oct30Cow2** (the start of the folder names). Run cells 1 → 7 in order without editing anything. If Colab disconnects, run 1 → 7 again: finished steps are skipped. **Do not run cell 8** (the pool): one person runs it once in `PipelineSteps.ipynb` when all three parts are done.

Needs: access to the BeastsAndBytes Shared Drive, and `HF_TOKEN` in Colab Secrets (key icon, notebook access on).

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install the model libraries.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline import PrepareClips, SaveTimelineSheet
from CalvingPipeline.Detect import DetectCows
from CalvingPipeline.Track import TrackCows
from CalvingPipeline.Crop import CropCows
from CalvingPipeline.Embed import EmbedCrops
from CalvingPipeline.Pool import BuildPool
from CalvingPipeline.Pipeline import AllRawVideos, ExpandAuto, StepDone, StepFolders, VideoStatus
from CalvingPipeline.Rename import CalvingName
from pathlib import Path
from CalvingPipeline.Paths import FindVideo, ListVideos, OutputRoot
import pandas as pd
from IPython.display import Image, Video, display
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

try:
    HfToken = userdata.get("HF_TOKEN")   # from Colab Secrets; never paste the token here
except Exception:
    HfToken = None
print("HF_TOKEN:", "found" if HfToken else "NOT SET - step 05 (DINOv3) will stop; add it in Colab Secrets")

In [ ]:
# 2. Settings: the only cell to edit.
display(ListVideos())    # the calving videos on the Shared Drive (HasOutputs = someone already ran this video)

# --- Which videos: one line each (file name, where the cow is). Output folder = the file name in the new
#     convention (CalvingPen3Nov11Cow1); old file names work too. VideoStem="..." sets another folder name. ---
# Small videos: run whole (ClipEnd=None). Sizes from Drive; ~0.9 GB = 1 h of raw video.
SmallVideos = [
    dict(VideoFile="CalvingPen2Oct29Cow3ROI3.mp4", ClipStart="00:00:00", ClipEnd=None),   # ~2 min ROI clip
    dict(VideoFile="CalvingPen3Sep26Cow1ROI2.mp4", ClipStart="00:00:00", ClipEnd=None),   # ~15 min ROI clip
    dict(VideoFile="CalvingPen2Sep26Cow2.mp4", ClipStart="00:00:00", ClipEnd=None),       # 27 min (the .avi is broken)
]
# Long videos (4.5 h): YOLO finds the cow and picks up to 3 windows of 10 min each (ClipStart="Auto").
LongVideos = [
    dict(VideoFile="CalvingPen1Nov11Cow1.avi", ClipStart="Auto"),
    dict(VideoFile="2026-11-12 11-46-30 Calving Pen 2 (2).avi", ClipStart="Auto"),   # -> CalvingPen2Nov12Cow2
    dict(VideoFile="2026-11-12 11-46-30 Calving Pen 2 (3).avi", ClipStart="Auto"),   # -> CalvingPen2Nov12Cow3
    dict(VideoFile="2026-11-12 11-46-30 Calving Pen 2 (5).avi", ClipStart="Auto"),   # -> CalvingPen2Nov12Cow5
    dict(VideoFile="2026-11-12 11-47-05 Calving Pen 3 (2).avi", ClipStart="Auto"),   # -> CalvingPen3Nov12Cow2
    dict(VideoFile="2026-11-12 11-47-05 Calving Pen 3 (4).avi", ClipStart="Auto"),   # -> CalvingPen3Nov12Cow4
    dict(VideoFile="2026-11-12 11-47-05 Calving Pen 3 (5).avi", ClipStart="Auto"),   # -> CalvingPen3Nov12Cow5
    dict(VideoFile="2026-10-29 09-48-06 Calving Pen 2 (1).avi", ClipStart="Auto"),   # -> CalvingPen2Oct29Cow1
    dict(VideoFile="2026-10-29 09-48-06 Calving Pen 2 (2).avi", ClipStart="Auto"),   # -> CalvingPen2Oct29Cow2
    dict(VideoFile="2026-10-29 09-48-06 Calving Pen 2 (4).avi", ClipStart="Auto"),   # -> CalvingPen2Oct29Cow4
    dict(VideoFile="2026-10-30 11-43-44 Calving Pen 2 (1).avi", ClipStart="Auto"),   # -> CalvingPen2Oct30Cow1
    dict(VideoFile="2026-10-30 11-43-44 Calving Pen 2 (2).avi", ClipStart="Auto"),   # -> CalvingPen2Oct30Cow2
]
# Already run (left out): CalvingPen2Sep26Cow1 (reference), CalvingPen3Sep26Cow1, CalvingPen2Oct29Cow3ROI1,
# CalvingPen3Nov11Cow1, CalvingPen2Nov11Cow1. Duplicates (left out): the .mp4 copies of Sep26 Pen1/Pen2,
# "2026-11-11 ... Pen 3 (2)/(3)" (= CalvingPen3Nov11Cow2/Cow3), "Copy of ...". Broken shortcuts: left out.
# Empty recordings (one picture, then nothing; size = only the empty-frame index): CalvingPen3Nov11Cow3,
# CalvingPen1Nov11Cow3, "2026-10-30 ... Pen 2 (4)", CalvingPen2Oct29Cow6.
# Not in the lists yet (header length is fake, part of the file has video): CalvingPen1Nov11Cow2, CalvingPen2Nov11Cow2,
# CalvingPen3Nov11Cow2, CalvingPen1Sep26Cow1.avi, CalvingPen3Sep26Cow2, "2026-10-29 ... (5)", "2026-10-30 ... (3)".
Batch = "Long"           # "Small", "Long" or "Both"
Videos = {"Small": SmallVideos, "Long": LongVideos, "Both": SmallVideos + LongVideos}[Batch]
# Videos = AllRawVideos()   # every readable raw calving video, windows found automatically (ClipStart="Auto")
Only = ['CalvingPen2Oct29', 'CalvingPen2Oct30Cow2']   # part 3 of 3: do not change
Redo = False             # False = skip steps already finished for a video; True = redo (overwrite) everything

# --- Automatic windows (only for videos with ClipStart="Auto") ---
ScanEverySeconds = 30    # 01: YOLO looks at one key frame every 30 s of the whole raw video (a few min per video)
WindowMinutes = 10       # 01: length of each window that is cut and tracked
AutoWindowCount = 3      # 01: up to this many windows per video, spread over the recording
MinCowShare = 0.8        # 01: a window needs a cow in view at least 80 % of the time

MakeTimeline = False     # 01: True = also save Timeline.jpg (a picture every 15 min) per video, to choose times by eye

# --- Test or full ---
Test = False             # True = only 4 minutes from TestStart ("a" folders); False = the whole clip ("b" folders)
TestStart = 1080         # seconds into the clip, only for Test = True

# --- Step settings: keep the defaults unless you are testing a change ---
TargetFps = 2            # 02: frames per second looked at (the camera updates ~2 times a second)
ImageSize = 1024         # 02: YOLO picture size (640 misses the lying cow far more often)
Confidence = 0.25        # 02: lowest YOLO confidence kept
SeedConfidence = 0.4     # 03: YOLO confidence needed to start following a new animal (the cow is carried over)
ChunkSeconds = 120       # 03: SAM's memory is restarted every 2 min (from where it left the cow)
Half = False             # 03: True = half precision on the GPU, faster; check it with Test = True first
Preview = True           # 02 (test only) and 03: preview videos
MinFrames, MinConfidence = 10, 0.4   # 04: drop tracks shorter than 5 s or less sure than 0.4
CropsPerTrack = 8 if Test else 24    # 04: at least this many crops per track
CropEverySeconds = 15    # 04: and at least one every 15 s
GroupSeconds = 60 if Test else 300   # 05: heatmap pieces of this many seconds
ExcludeSeconds = 60 if Test else 300 # 06: self-check may match the same track this far away

# --- Folders (nothing to edit) ---
AutoSettings = dict(EverySeconds=ScanEverySeconds, WindowMinutes=WindowMinutes, Windows=AutoWindowCount, MinCowShare=MinCowShare)
Run = ExpandAuto(Videos, OutputRoot, AllowScan=False, **AutoSettings)   # Auto videos already scanned -> their windows
Run = [dict(V, VideoStem=V.get("VideoStem") or Path(CalvingName(Path(V["VideoFile"]).name)[0]).stem) for V in Run]
Run = [V for V in Run if Only is None or any(V["VideoStem"].startswith(O) for O in Only)]
Folder = {Step: Name for Step, Name in StepFolders(Test).items()}
Start, End = (TestStart, TestStart + 240) if Test else (0, None)
def Dirs(V):
    VideoDir = OutputRoot / V["VideoStem"]
    return VideoDir, VideoDir / "ReferenceClip.mp4"
def Skip(Step, V, **Settings):
    if V.get("NeedsScan"):
        print(f"{V['VideoStem']}: not scanned yet (ClipStart='Auto'): run cell 3 first")
        return True
    if not Redo and StepDone(Step, Dirs(V)[0], Test, **Settings):
        print(f"{V['VideoStem']}: step 0{Step} already done, skipped (Redo = True to redo it)")
        return True
    print(f"\n=== {V['VideoStem']}: step 0{Step} ===")
    return False
for V in Run:
    print(V["VideoStem"], "| auto, scan in cell 3" if V.get("NeedsScan") else
          f"| {V.get('ClipStart')}-{V.get('ClipEnd')} | clip {'cut' if Dirs(V)[1].exists() else 'not cut yet'}",
          "| outputs:", Dirs(V)[0])

In [ ]:
# 3. Step 01 · Cut the clip (ffmpeg), for each video. ClipStart="Auto": first YOLO scans the whole video
#    and picks windows (Scan/CowTimeline.png); each window becomes its own run (<name>Win1, Win2, ...).
Scanned = [V for V in Run if V.get("NeedsScan")]
if Scanned:
    Run = [V for V in Run if not V.get("NeedsScan")] + ExpandAuto(
        [dict(V, ClipStart="Auto") for V in Scanned], OutputRoot, AllowScan=True, **AutoSettings)
    for V in Scanned:
        display(Image(str(OutputRoot / V["VideoStem"] / "Scan" / "CowTimeline.png"), width=1100))
    print("Runs now:", [V["VideoStem"] for V in Run])
for V in Run:
    VideoDir, ClipPath = Dirs(V)
    if Skip(1, V):
        continue
    VideoPath = FindVideo(V["VideoFile"])
    if MakeTimeline:
        SaveTimelineSheet(VideoPath, VideoDir / "Timeline.jpg", EveryMinutes=15)
        display(Image(str(VideoDir / "Timeline.jpg"), width=1200))
    Summary = PrepareClips(VideoPath, VideoDir, V.get("ClipStart", 0), V.get("ClipEnd"), Rebuild=V.get("Rebuild", "Auto"))
    Clip = Summary["ReferenceClip"]
    print(f"Clip {Summary['ClipStart']}-{Summary['ClipEnd']}: {Clip['Duration']}, {Clip['Fps']} fps, "
          f"{Clip['Width']}x{Clip['Height']}  (check: length as expected; 30 fps for raw videos, 2 fps for 2FPS ROI clips)")

In [ ]:
# 4. Step 02 · Detect cows (YOLO26n), for each video.
for V in Run:
    VideoDir, ClipPath = Dirs(V)
    if Skip(2, V, TargetFps=TargetFps, ImageSize=ImageSize, Confidence=Confidence, StartSeconds=Start):
        continue
    DetectCows(ClipPath, VideoDir / Folder[2], Start, End, TargetFps=TargetFps, Confidence=Confidence,
               ImageSize=ImageSize, Preview=Preview and Test)
    Summary = pd.read_csv(VideoDir / Folder[2] / "DetectionSummary.csv")
    print(f"cow seen in {Summary.FramesWithCow.sum() / Summary.SampledFrames.sum():.0%} of frames; "
          f"minutes with CowShare < 0.5: {list(Summary.ClipTime[Summary.CowShare < 0.5])}")

In [ ]:
# 5. Step 03 · Track the cows (SAM 2.1, video mode), for each video. Rerun to continue a stopped run.
for V in Run:
    VideoDir, ClipPath = Dirs(V)
    print(f"\n=== {V['VideoStem']}: step 03 ===")
    TrackCows(ClipPath, VideoDir / Folder[2], VideoDir / Folder[3], ChunkSeconds=ChunkSeconds,
              SeedConfidence=SeedConfidence, CompareTracksPath=VideoDir / "BaselineByteTrack" / "Tracks.csv",
              Half=Half, Preview=Preview, RedoFinished=Redo)
    display(pd.read_csv(VideoDir / Folder[3] / "TrackSummary.csv"))
    print("Preview (open in Drive):", VideoDir / Folder[3] / "PreviewTracks.mp4")

In [ ]:
# 6. Step 04 · Cut out the cows (SAM 2.1, image mode): masked crops, for each video.
for V in Run:
    VideoDir, ClipPath = Dirs(V)
    if Skip(4, V):
        continue
    CropCows(ClipPath, VideoDir / Folder[3] / "Tracks.csv", VideoDir / Folder[4], MinFrames=MinFrames,
             MinConfidence=MinConfidence, CropsPerTrack=CropsPerTrack, CropEverySeconds=CropEverySeconds)
    display(pd.read_csv(VideoDir / Folder[4] / "KeptTracks.csv"))
    display(Image(str(VideoDir / Folder[4] / "Sheets" / "Overview.jpg")))

In [ ]:
# 7. Step 05 · Fingerprint each crop (DINOv3), for each video. No DINOv2 fallback: stops if the token fails.
for V in Run:
    VideoDir, ClipPath = Dirs(V)
    if Skip(5, V):
        continue
    EmbedCrops(VideoDir / Folder[4], VideoDir / Folder[5], V["VideoStem"], Token=HfToken, FallbackModelId=None,
               GroupSeconds=GroupSeconds)
    display(Image(str(VideoDir / Folder[5] / "TrackSimilarity.png")))

In [ ]:
raise SystemExit("Part notebooks do not build the pool: run cell 8 once in PipelineSteps.ipynb when parts 1-3 are done.")
# 8. Step 06 · Pool ALL video folders' fingerprints and self-check (no model). Run once all videos passed cell 7.
# Cows are labelled Pen1Cow1, Pen1Cow2, Pen2Cow1, ... A video with two cows: in its Identities.csv (made on the first run) write Cow2 for the second cow's tracks, then rerun.
Stats = BuildPool(OutputRoot, OutputRoot / Folder[6], EmbeddingFolder=Folder[5], ExcludeSeconds=ExcludeSeconds)
display(pd.read_csv(OutputRoot / Folder[6] / "PoolSummary.csv"))
if (OutputRoot / Folder[6] / "SelfCheck.png").exists():
    display(Image(str(OutputRoot / Folder[6] / "SelfCheck.png")))
display(VideoStatus(OutputRoot, Test))

## About this notebook

Each cell calls one function from `CalvingPen/CalvingPipeline/`: `PrepareClips` (VideoTools.py), `DetectCows` (Detect.py), `TrackCows` (Track.py), `CropCows` (Crop.py), `EmbedCrops` (Embed.py), `BuildPool` (Pool.py). What each model does, what we found, and the glossary: `CalvingPen/README.md`.

**Check before moving on:** step 02 `CowShare` per minute (low = YOLO cannot see her); step 03 few long tracks and low `YoloDisagrees`, masks on the cow in the preview; step 04 `Overview.jpg` shows only the cow (no calf, no floor); step 05 bright blocks in the heatmap. Step 06 labels come from each video's `Identities.csv` (edit by hand; never overwritten).